# Flask

Flask expose une API REST. Ce notebook écrit l'API de panier (GET, POST, PATCH, DELETE) et l'appelle en local.
Le serveur de développement écoute sur `127.0.0.1`, port 5000. Aucun prérequis cloud.
Le panier est initialisé avec le produit `je8zng` pour pouvoir tester PATCH et DELETE.


## Lancement local

`start_server` relance Flask sans bloquer le notebook. `FLASK_APP` pointe vers `/tmp/server.py`.


In [ ]:
import os
import signal
import subprocess
import time

server = None

def stop_server():
    if server:
        os.killpg(server.pid, signal.SIGTERM)
        server.terminate()

def start_server():
    # Petite astuce pour exécuter le serveur sans quitter le notebook
    stop_server()
    time.sleep(1.5)
    print("Serveur prêt")
    return subprocess.Popen("FLASK_APP=/tmp/server.py flask run", shell=True, preexec_fn=os.setsid)


## API panier


In [ ]:
%%writefile /tmp/server.py
from flask import Flask, request, jsonify

app = Flask(__name__)

cart = [{
    'id': "je8zng",
    'quantity': 3
}]

def check_fields(body, fields):
    required_parameters_set = set(fields)
    fields_set = set(body.keys())
    return required_parameters_set <= fields_set

@app.route('/')
def hello_world():
    return "Coucou !"

@app.route('/cart', methods=['GET'])
def list_cart():
    return jsonify(cart), 200

@app.route('/cart', methods=['POST'])
def add_to_cart():
    try:
        body = request.get_json()
        if not check_fields(body, {'id', 'quantity'}):
            return jsonify({'error': "Missing fields."}), 400

        for i, item in enumerate(cart):
            if item['id'] == body.get('id', ""):
                cart[i]['quantity'] += int(body.get('quantity', 0))
                return jsonify({}), 200

        cart.append(body)
        return jsonify({}), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/cart', methods=['PATCH'])
def edit_cart():
    try:
        body = request.get_json()
        if not check_fields(body, {'id', 'quantity'}):
            return jsonify({'error': "Missing fields."}), 400

        for i, item in enumerate(cart):
            if item['id'] == body.get('id', ""):
                cart[i]['quantity'] = int(body.get('quantity', 0))
                return jsonify({}), 200

        return jsonify({'error': "Product not found."}), 404
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/cart', methods=['DELETE'])
def remove_from_cart():
    try:
        body = request.get_json()
        if not check_fields(body, {'id'}):
            return jsonify({'error': "Missing fields."}), 400

        for i, item in enumerate(cart):
            if item['id'] == body['id']:
                del cart[i]
                return jsonify({}), 200

        return jsonify({'error': "Product not found."}), 404
    except Exception as e:
        return jsonify({'error': str(e)}), 500


## Appels


In [ ]:
!pip install requests -q
import requests

server = start_server()


In [ ]:
req = requests.post("http://127.0.0.1:5000/cart", json={
    'id': "je8zng"
})
print(req.status_code, req.json())


In [ ]:
req = requests.post("http://127.0.0.1:5000/cart", json={
    'id': "je8zng",
    'quantity': 1
})
print(req.status_code, req.json())


In [ ]:
requests.get("http://127.0.0.1:5000/cart").json()


In [ ]:
req = requests.post("http://127.0.0.1:5000/cart", json={
    'id': "je8zng",
    'quantity': 2
})
print(req.status_code, req.json())


In [ ]:
requests.get("http://127.0.0.1:5000/cart").json()


In [ ]:
req = requests.patch("http://127.0.0.1:5000/cart", json={
    'id': "aaaaa",
    'quantity': 10
})
print(req.status_code, req.json())


In [ ]:
req = requests.patch("http://127.0.0.1:5000/cart", json={
    'id': "je8zng",
    'quantity': 10
})
print(req.status_code, req.json())


In [ ]:
requests.get("http://127.0.0.1:5000/cart").json()


In [ ]:
req = requests.delete("http://127.0.0.1:5000/cart", json={
    'id': "je8zng"
})
print(req.status_code, req.json())


In [ ]:
requests.get("http://127.0.0.1:5000/cart").json()


In [ ]:
stop_server()
